In [2]:
# Conformal prediction + newsvendor cost 

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
np.random.seed(42)

# --- Config ---
FORECAST_LENGTH = 14
CONTEXT_LENGTH = 90
MIN_SERIES_LENGTH = CONTEXT_LENGTH + FORECAST_LENGTH + 2

DATA_PATH = Path("../data/M4")
TRAIN_FILE = DATA_PATH / "Daily-train.csv"
TEST_FILE = DATA_PATH / "Daily-test.csv"
RESULTS_DIR = Path("../results")

N_SERIES_CALIB = 500
N_SERIES_TEST = 1000

COST_UNDERAGE = 2.0
COST_OVERAGE = 1.0
CRITICAL_RATIO = COST_UNDERAGE / (COST_UNDERAGE + COST_OVERAGE)
COVERAGE_LEVEL = 0.90

MODEL_NAMES = [
    "ChronosBoltTiny", "TTM", "TimesFM3",
    "LightGBM", "XGBoost", "RandomForest",
    "NHiTS", "DLinear", "TiDE",
]

print(f"tau={CRITICAL_RATIO:.3f} | calib={N_SERIES_CALIB} | test={N_SERIES_TEST} | modelli={MODEL_NAMES}")

# --- Data loading + canonical split ---
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

id_col = train_df.columns[0]
train_value_cols = train_df.columns[1:]
test_value_cols = test_df.columns[1:]

train_df[id_col] = train_df[id_col].astype(str)
test_df[id_col] = test_df[id_col].astype(str)

lengths = train_df[train_value_cols].notna().sum(axis=1)
train_df_f = train_df[lengths >= MIN_SERIES_LENGTH].reset_index(drop=True)
test_df_f = test_df.set_index(id_col).loc[train_df_f[id_col]].reset_index()

calib_ids = train_df_f[id_col].iloc[:N_SERIES_CALIB].tolist()
test_ids = train_df_f[id_col].iloc[N_SERIES_CALIB:N_SERIES_CALIB + N_SERIES_TEST].tolist()

train_lookup = train_df_f.set_index(id_col)
test_lookup = test_df_f.set_index(id_col)

def get_train_values(uid):
    return train_lookup.loc[uid, train_value_cols].dropna().values.astype(float)

def get_test_values(uid):
    return test_lookup.loc[uid, test_value_cols].dropna().values.astype(float)

print(f"series after filter: {len(train_df_f)} | calib: {len(calib_ids)} | test: {len(test_ids)}")

# --- Load pre-computed forecasts from CSV ---
def load_forecast_csv(model_name, split, expected_ids):
    path = RESULTS_DIR / f"{split}_{model_name}.csv"
    df = pd.read_csv(path)
    uid_col = df.columns[0]
    h_cols = sorted([c for c in df.columns if c.startswith("h")], key=lambda c: int(c[1:]))
    df[uid_col] = df[uid_col].astype(str)
    forecasts = {row[uid_col]: row[h_cols].values.astype(float) for _, row in df.iterrows()}
    missing = set(expected_ids) - set(forecasts.keys())
    if missing:
        print(f"  WARNING [{model_name}/{split}]: {len(missing)} id mancanti (es. {list(missing)[:3]})")
    return forecasts

loaded_calib = {name: load_forecast_csv(name, "calib", calib_ids) for name in MODEL_NAMES}
loaded_test = {name: load_forecast_csv(name, "test", test_ids) for name in MODEL_NAMES}

# --- Step 1: calibration residuals ---
residuals = {name: [] for name in MODEL_NAMES}
for uid in tqdm(calib_ids, desc="calibration"):
    y_true = get_train_values(uid)[-FORECAST_LENGTH:]
    for name in MODEL_NAMES:
        residuals[name].append(y_true - loaded_calib[name][uid])
residuals = {name: np.array(vals) for name, vals in residuals.items()}  # (n_series, 14)

# --- Step 2: conformal quantiles ---
def conformal_quantile(residual_values, tau):
    r = np.sort(np.asarray(residual_values, dtype=float))
    n = len(r)
    q_level = min(np.ceil((n + 1) * tau) / n, 1.0)
    return np.quantile(r, q_level, method="higher")

q_abs = {
    name: np.array([conformal_quantile(np.abs(residuals[name][:, h]), COVERAGE_LEVEL)
                     for h in range(FORECAST_LENGTH)])
    for name in MODEL_NAMES
}
q_signed = {
    name: np.array([conformal_quantile(residuals[name][:, h], CRITICAL_RATIO)
                     for h in range(FORECAST_LENGTH)])
    for name in MODEL_NAMES
}

# --- Step 3: held-out test forecasts + actuals ---
test_forecasts = {uid: {name: loaded_test[name][uid] for name in MODEL_NAMES} for uid in test_ids}
test_actuals = {uid: get_test_values(uid) for uid in test_ids}

# --- Step 4: coverage check ---
coverage_records = []
for name in MODEL_NAMES:
    covered = np.array([np.abs(test_actuals[uid] - test_forecasts[uid][name]) <= q_abs[name]
                         for uid in test_ids])
    coverage_records.append({
        "model": name,
        "coverage_overall": covered.mean(),
        "coverage_h1": covered[:, 0].mean(),
        "coverage_h14": covered[:, -1].mean(),
    })
coverage_df = pd.DataFrame(coverage_records)
print("\n" + coverage_df.to_string(index=False))

# --- Step 5: newsvendor cost ---
def newsvendor_cost(y_true, order, cu, co):
    shortage = np.maximum(y_true - order, 0)
    surplus = np.maximum(order - y_true, 0)
    return cu * shortage + co * surplus

cost_records = []
for name in MODEL_NAMES:
    cost_naive, cost_conformal = 0.0, 0.0
    for uid in test_ids:
        y_true = test_actuals[uid]
        point_fc = test_forecasts[uid][name]
        cost_naive += newsvendor_cost(y_true, point_fc, COST_UNDERAGE, COST_OVERAGE).sum()
        cost_conformal += newsvendor_cost(y_true, point_fc + q_signed[name],
                                           COST_UNDERAGE, COST_OVERAGE).sum()
    n_obs = len(test_ids) * FORECAST_LENGTH
    cost_records.append({
        "model": name,
        "cost_naive_avg": cost_naive / n_obs,
        "cost_conformal_avg": cost_conformal / n_obs,
        "cost_reduction_pct": 100 * (1 - cost_conformal / cost_naive),
    })
cost_df = pd.DataFrame(cost_records).sort_values("cost_conformal_avg")
print("\n" + cost_df.to_string(index=False))

# --- Step 6: accuracy + cost, side by side ---
def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    denom = np.where((np.abs(y_true) + np.abs(y_pred)) == 0, 1e-8, (np.abs(y_true) + np.abs(y_pred)) / 2)
    return 100.0 * np.mean(np.abs(y_true - y_pred) / denom)

def mase(y_true, y_pred, train_values, seasonality=1):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    train_values = np.asarray(train_values, dtype=float)
    if len(train_values) <= seasonality:
        return np.nan
    naive_mae = np.mean(np.abs(np.diff(train_values, n=seasonality)))
    return np.mean(np.abs(y_true - y_pred)) / max(naive_mae, 1e-8)

acc_records = []
for name in MODEL_NAMES:
    smapes = [smape(test_actuals[uid], test_forecasts[uid][name]) for uid in test_ids]
    mases = [mase(test_actuals[uid], test_forecasts[uid][name], get_train_values(uid)) for uid in test_ids]
    acc_records.append({"model": name, "smape_mean": np.mean(smapes), "mase_mean": np.mean(mases)})

final_df = pd.DataFrame(acc_records).merge(cost_df, on="model").sort_values("smape_mean")
print("\n" + final_df.to_string(index=False))

tau=0.667 | calib=500 | test=1000 | modelli=['ChronosBoltTiny', 'TTM', 'TimesFM3', 'LightGBM', 'XGBoost', 'RandomForest', 'NHiTS', 'DLinear', 'TiDE']
series after filter: 4211 | calib: 500 | test: 1000


calibration: 100%|██████████| 500/500 [00:00<00:00, 586.13it/s]



          model  coverage_overall  coverage_h1  coverage_h14
ChronosBoltTiny          0.861000        0.875         0.900
            TTM          0.838857        0.810         0.871
       TimesFM3          0.890929        0.836         0.924
       LightGBM          0.842571        0.811         0.865
        XGBoost          0.842500        0.810         0.862
   RandomForest          0.829857        0.834         0.835
          NHiTS          0.884143        0.833         0.907
        DLinear          0.831500        0.788         0.837
           TiDE          0.882714        0.767         0.903

          model  cost_naive_avg  cost_conformal_avg  cost_reduction_pct
           TiDE      215.396669          196.746393            8.658572
          NHiTS      199.539404          199.401781            0.068970
       TimesFM3      238.642856          215.558633            9.673125
ChronosBoltTiny      255.726918          230.536229            9.850621
            TTM      270.733